In [1]:
import sys
from pathlib import Path

ruta_raiz = Path.cwd().parent

ruta_de_src = Path(ruta_raiz / "src")

sys.path.append(str(ruta_de_src))

from insercion_registros import * # Funciones del ejercicio 4
from dataset_reader import obtener_columnas  # Función para reutilizar en el ejercicio b

# Rutas
ruta_iadiza = Path(ruta_raiz / "raw_datasets" / "Colección Ornitológica del IADIZA-CCT CONICET Mendoza IADIZA-COI" / "occurrence.txt")

ruta_procesado = Path(ruta_raiz / "processed_datasets" / "IADIZA" / "occurrence.txt")

nombre_dataset = "IADIZA"
#Nombre del dataset para registrar la operación en el log

Ejercicio 4.A - Definir estructura

In [2]:
registro_ejemplo = crear_estructura_registro(
    license="CC_BY_NC_4_0",
    institutionCode="IADIZA",
    collectionCode="COI",
    basisOfRecord="PRESERVED_SPECIMEN",
    occurrenceStatus="PRESENT",
    recordedBy="Juan Pérez",
    eventDate="2024-03-15",
    year="2024",
    month="3",
    day="15",
    countryCode="AR",
    stateProvince="Mendoza",
    locality="Reserva Natural Ñacuñán",
    decimalLatitude="-34.0412",
    decimalLongitude="-67.9587",
    scientificName="Milvago chimango (Vieillot, 1816)",
    kingdom="Animalia",
    phylum="Chordata",
    taxonClass="Aves",
    order="Falconiformes",
    family="Falconidae",
    genus="Milvago",
    specificEpithet="chimango",
    taxonRank="SPECIES",
)
print("Registro de ejemplo:")
for i, j in registro_ejemplo.items():
    print(f"  {i}: {j}")

Registro de ejemplo:
  license: CC_BY_NC_4_0
  modified: 
  institutionCode: IADIZA
  collectionCode: COI
  basisOfRecord: PRESERVED_SPECIMEN
  occurrenceStatus: PRESENT
  recordedBy: Juan Pérez
  catalogNumber: 
  eventDate: 2024-03-15
  year: 2024
  month: 3
  day: 15
  countryCode: AR
  stateProvince: Mendoza
  county: 
  locality: Reserva Natural Ñacuñán
  decimalLatitude: -34.0412
  decimalLongitude: -67.9587
  coordinateUncertaintyInMeters: 
  scientificName: Milvago chimango (Vieillot, 1816)
  kingdom: Animalia
  phylum: Chordata
  class: Aves
  order: Falconiformes
  family: Falconidae
  genus: Milvago
  specificEpithet: chimango
  taxonRank: SPECIES
  vernacularName: 


Ejercicio 4.B - Genera un registro vacío, donde todos los valores estén inicializados.

In [3]:
# Obtener columnas del dataset
columnas = obtener_columnas(ruta_iadiza)

# Si obtener_columnas devuelve string entonces lo convertimos a lista separando por tabulaciones
if len(columnas) == 1:
    columnas = columnas[0].split("\t")

# Crear registro vacío
registro_vacio = crear_registro_vacio(columnas)

print("Registro vacío:")
for k, v in registro_vacio.items():
    print(f"{k}: {v}")

Registro vacío:
gbifID: 
accessRights: 
bibliographicCitation: 
language: 
license: 
modified: 
publisher: 
references: 
rightsHolder: 
type: 
institutionID: 
collectionID: 
datasetID: 
institutionCode: 
collectionCode: 
datasetName: 
ownerInstitutionCode: 
basisOfRecord: 
informationWithheld: 
dataGeneralizations: 
dynamicProperties: 
occurrenceID: 
catalogNumber: 
recordNumber: 
recordedBy: 
recordedByID: 
individualCount: 
organismQuantity: 
organismQuantityType: 
sex: 
lifeStage: 
reproductiveCondition: 
caste: 
behavior: 
vitality: 
establishmentMeans: 
degreeOfEstablishment: 
pathway: 
georeferenceVerificationStatus: 
occurrenceStatus: 
preparations: 
disposition: 
associatedOccurrences: 
associatedReferences: 
associatedSequences: 
associatedTaxa: 
otherCatalogNumbers: 
occurrenceRemarks: 
organismID: 
organismName: 
organismScope: 
associatedOrganisms: 
previousIdentifications: 
organismRemarks: 
materialEntityID: 
materialEntityRemarks: 
verbatimLabel: 
materialSampleID: 
even

Ejercicio 4.C - Validación de registros, antes de ser insertados

In [4]:
# Registro válido
es_valido, errores = validar_registro(registro_ejemplo)
print(f"¿Registro válido? {es_valido}")
if errores:
    print("Errores:", errores)

# Registro con errores para probar las validaciones
registro_invalido = crear_estructura_registro(
    decimalLatitude="999",                    # fuera de rango
    decimalLongitude="-50.0",
    countryCode="ARGENTINA",                  # código inválido
    occurrenceStatus="MAYBE",                 # valor no permitido
    eventDate="2099-01-01",                   # fecha futura
    coordinateUncertaintyInMeters="-5",       # valor negativo
)
es_valido2, errores2 = validar_registro(registro_invalido)
print(f"\n¿Registro inválido detectado? {not es_valido2}")
print("Errores encontrados:")
for e in errores2:
    print(f"  - {e}")

¿Registro válido? True

¿Registro inválido detectado? True
Errores encontrados:
  - decimalLatitude inválida: '999'. Debe estar entre -90 y 90
  - eventDate inválida: '2099-01-01'. La fecha no puede ser futura
  - countryCode inválido: 'ARGENTINA' Valores permitidos: {'EC', 'PY', 'BO', 'PE', 'CL', 'BR', 'CO', 'VE', 'AR', 'UY'} 
  - coordinateUncertaintyInMeters inválida: '-5' Debe ser un número entre 0 y 100
  - occurrenceStatus inválido: 'MAYBE' Valores permitidos: {'ABSENT', 'PRESENT'} 


Ejercicio 4.D - Preparar estructura para añadir al CSV.

In [5]:
columnas = obtener_columnas(ruta_iadiza)

# Si obtener_columnas devuelve string entonces lo convertimos a lista separando por tabulaciones
if len(columnas) == 1:
    columnas = columnas[0].split("\t")

registro_listo = preparar_estructura(registro_ejemplo, ruta_iadiza, columnas, nombre_dataset)
print(f"occurrenceID generado : {registro_listo['occurrenceID']}")
print(f"catalogNumber generado: {registro_listo['catalogNumber']}")
print(f"gbifID                : '{registro_listo['gbifID']}' (asignado por GBIF externamente)")

print("\nRegistro completo listo para el CSV:")
for k, v in registro_listo.items():
    if v != "":  # No imprime los campos vacíos para que sea más legible
        print(f"  {k}: {v}")

occurrenceID generado : IADIZA:COI:007009
catalogNumber generado: 007009
gbifID                : '' (asignado por GBIF externamente)

Registro completo listo para el CSV:
  license: CC_BY_NC_4_0
  institutionCode: IADIZA
  collectionCode: COI
  basisOfRecord: PRESERVED_SPECIMEN
  occurrenceID: IADIZA:COI:007009
  catalogNumber: 007009
  recordedBy: Juan Pérez
  occurrenceStatus: PRESENT
  eventDate: 2024-03-15
  year: 2024
  month: 3
  day: 15
  countryCode: AR
  stateProvince: Mendoza
  locality: Reserva Natural Ñacuñán
  decimalLatitude: -34.0412
  decimalLongitude: -67.9587
  scientificName: Milvago chimango (Vieillot, 1816)
  kingdom: Animalia
  phylum: Chordata
  class: Aves
  order: Falconiformes
  family: Falconidae
  genus: Milvago
  specificEpithet: chimango
  taxonRank: SPECIES


Ejercicio 4.F - Agregar un registro con datos ingresados por teclado en un nuevo archivo conservando la estructura original

In [6]:
#Crea un nuevo registro ingresado por teclado
registro = ingresar_registro_por_teclado()

#Valida si el nuevo registro es correcto antes de intentar insertarlo
es_valido, errores = validar_registro (registro)
if es_valido:
    insertar_registro(ruta_iadiza, ruta_procesado, registro, nombre_dataset)
    #Si el registro es válido se inserta
else:
    print ("Errores:")
    for e in errores:
        print ("-", e)
    #Si el registro no es válido se muestran los errores y no se inserta

Ingrese los datos del nuevo registro:


License:  yeujs
Institution Code:  1274
Collection Code:  273
Basis of Record:  ueormx
Occurrence Status (PRESENT/ABSENT):  PRESENT
Recorded By:  uowjemd
Event Date (YYYY-MM-DD):  2007-03-02
Year:  2007
Month:  03
Day:  02
Country Code:  AR
State/Province:  Buenos Aires
Locality:  La Plata
Latitude:  35
Longitude:  74
Scientific Name:  jwusos


Registro insertado correctamente


Ejercicio 4.G - Inserta múltiples registros ingresados por teclado en una sola operación

In [ ]:
insertar_multiples_registros(ruta_iadiza, ruta_procesado, nombre_dataset)

Carga de múltiples registros (dejar vacío 'scientificName' para terminar)
Ingrese los datos del nuevo registro:


License:  yeirumxx
Institution Code:  1673
Collection Code:  723
Basis of Record:  ueodma
Occurrence Status (PRESENT/ABSENT):  ABSENT
Recorded By:  uwieslc
Event Date (YYYY-MM-DD):  2009-07-01
Year:  2009
Month:  07
Day:  01
Country Code:  AR
State/Province:  Buenos Aires
Locality:  La Plata
Latitude:  38
Longitude:  62
Scientific Name:  uwomex


Registro agregado

Ingrese los datos del nuevo registro:


License:  ueorjc
Institution Code:  2674
Collection Code:  762
Basis of Record:  uwoiruck
Occurrence Status (PRESENT/ABSENT):  PRESENT
Recorded By:  uqiejda
Event Date (YYYY-MM-DD):  2016-09-15
